# Parte I · Fase 1 — Mapeamento Estético, Coordenadas e Cor
**Programação para Visualização de Dados · MEI IPCA/EST · 2026/2027**

Este notebook percorre, de ponta a ponta, tudo o que foi implementado na Parte I, Fase 1
(Caps. 1–5 de Wilke, *Fundamentals of Data Visualization*):

1. Dados e dicionário de variáveis
2. Mapeamento estético (posição, forma, tamanho, matiz) e escalas linear vs. log
3. Paletas qualitativa, sequencial e divergente
4. **Nó de auditoria de cor** (requisito LangChain/LangGraph)
5. Grafo cíclico LangGraph: gerador → auditor → executor
6. Cenários de autocorreção com o LLM real

> Todas as figuras são geradas por código; não há edição manual.
> Requer `GOOGLE_API_KEY` no ficheiro `.env` (as secções 5 e 6 chamam o Gemini).

In [ ]:
import json, os, sys
from pathlib import Path

# Garante que o notebook corre a partir da raiz do projeto (imports de `src`).
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_colwidth", 90)
print("Raiz do projeto:", ROOT)

## 1. Dados e dicionário de variáveis
Dataset **Gapminder** (1704 linhas: 142 países × 12 anos). `src/data_prep.py` valida o ficheiro bruto,
grava a versão limpa e gera um **perfil** com a *natureza* de cada variável. Esse perfil é o que o
auditor consulta para decidir se uma paleta é adequada.

In [ ]:
from src import data_prep
df = data_prep.load_clean()
data_prep.main()                      # regrava gapminder_clean.csv e variable_profile.json
display(df.head())
profile = json.loads(data_prep.PROFILE.read_text(encoding="utf-8"))
display(pd.DataFrame(profile).T[["nature", "cardinality", "min", "max", "skew", "suggest_log_scale"]])

**Leitura:** `pop` (assimetria 8,3) e `gdpPercap` (3,9) têm cauda longa à direita → candidatas a **escala logarítmica**.
`lifeExp` é aproximadamente simétrica → escala linear. `continent` e `country` são nominais; `year` é ordenada.

## 2. Mapeamento estético e escalas (Caps. 2–3)
As figuras base F1–F6 estão em `src/plots.py`. Cada uma demonstra uma decisão de design.

In [ ]:
from src import plots
d = plots._data()
p1 = plots.f1_position_hue_size(d)
display(Image(str(p1)))

**F1.** Quatro variáveis num gráfico: PIB e esperança de vida em **posição** (o canal mais preciso), continente em **matiz**
(qualitativa) e população em **tamanho**. O tamanho é proporcional à **área** do círculo, não ao raio.

In [ ]:
p2 = plots.f2_linear_vs_log(d)
display(Image(str(p2)))

**F2 — porquê escala log?** Em escala linear (esquerda) a maioria dos países fica comprimida junto ao eixo y
e a relação parece uma curva de saturação. Em escala log (direita) a relação é aproximadamente linear e todos
os países ficam legíveis. A escolha justifica-se pela assimetria do PIB (3,85) e porque as diferenças relevantes são *multiplicativas*.

In [ ]:
p3 = plots.f3_shape_redundant(d)
display(Image(str(p3)))

**F3 — codificação redundante.** O continente é codificado por **forma e cor**. Se a cor falhar (daltonismo, impressão a P&B), a forma continua a distinguir os grupos.

## 3. Paletas de cor (Cap. 4)
A regra: **o tipo de paleta tem de corresponder à natureza da variável.**

| Natureza | Paleta | Exemplo |
|---|---|---|
| categórica nominal | qualitativa (Okabe-Ito) | continente |
| contínua / ordenada | sequencial (viridis) | ano |
| contínua com centro com significado | divergente (RdBu + `TwoSlopeNorm`) | desvio à mediana |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from src import palettes

fig, axes = plt.subplots(3, 1, figsize=(8, 2.6))
# qualitativa
for i, (c, col) in enumerate(palettes.QUALITATIVE.items()):
    axes[0].add_patch(plt.Rectangle((i, 0), 1, 1, color=col)); axes[0].text(i + .5, .5, c, ha="center", va="center", color="white", fontsize=8)
axes[0].set_xlim(0, 5); axes[0].set_ylim(0, 1); axes[0].set_title("Qualitativa (Okabe-Ito)", fontsize=9, loc="left")
# sequencial / divergente
grad = [[i / 255 for i in range(256)]]
axes[1].imshow(grad, aspect="auto", cmap=palettes.SEQUENTIAL); axes[1].set_title("Sequencial (viridis)", fontsize=9, loc="left")
axes[2].imshow(grad, aspect="auto", cmap=palettes.DIVERGING); axes[2].set_title("Divergente (RdBu), centro no meio", fontsize=9, loc="left")
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
p4 = plots.f4_sequential(d); p5 = plots.f5_diverging(d)
display(Image(str(p4))); display(Image(str(p5)))

**F4 — sequencial.** O ano é uma variável ordenada: escuro → claro = antigo → recente, com barra de cor.

**F5 — divergente.** A cor mostra o desvio da esperança de vida face à **mediana mundial** (71,9 anos). O branco fica exatamente no centro (desvio 0) graças a `TwoSlopeNorm`:
vermelho = abaixo da mediana, azul = acima. Uma divergente sem centro definido enganaria o leitor — é uma das regras do auditor.

## 4. Nó de auditoria de cor
`src/agents/audit.py` **inspeciona o código gerado sem o executar** (análise estática com `ast`):
identifica a variável mapeada em cor (`c=`, `hue=`), identifica a paleta (`cmap=`, `palette=`) e cruza com a natureza da variável.

| Regra | Condição | Resultado |
|---|---|---|
| R1 | contínua + qualitativa | rejeita |
| R2 | nominal + sequencial/divergente | rejeita |
| R3 | ordinal + qualitativa | aviso |
| R4 | divergente sem centro (`norm`/`center`) | rejeita |
| R5 | arco-íris (`jet`, `rainbow`…) | rejeita |
| R6 | qualitativa com > 8 categorias | aviso |
| R7 | erro de sintaxe | rejeita |

In [ ]:
from src.agents.audit import static_audit

cases = {
    "contínua + tab10":            "ax.scatter(d.x, d.y, c=d.lifeExp, cmap='tab10')",
    "nominal + viridis":           "ax.scatter(d.x, d.y, c=d.continent, cmap='viridis')",
    "divergente sem centro":       "ax.scatter(d.x, d.y, c=d.lifeExp, cmap='RdBu')",
    "arco-íris (jet)":             "ax.scatter(d.x, d.y, c=d.pop, cmap='jet')",
    "país com 142 cores":          "sns.scatterplot(data=d, hue='country', palette='tab20')",
    "contínua + viridis (OK)":     "ax.scatter(d.x, d.y, c=d.lifeExp, cmap='viridis')",
    "nominal + QUALITATIVE (OK)":  "ax.scatter(d.x, d.y, c=d.continent.map(QUALITATIVE))",
    "divergente com centro (OK)":  "ax.scatter(d.x, d.y, c=d.lifeExp, cmap='RdBu', norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))",
}
rows = []
for name, snippet in cases.items():
    r = static_audit(snippet, profile)
    rows.append({"caso": name, "aprovado": "✅" if r.approved else "❌",
                 "regras": ", ".join(f"{f.rule}/{f.severity}" for f in r.findings) or "-"})
display(pd.DataFrame(rows))

In [ ]:
# O feedback que o auditor devolve ao agente gerador (para reescrita):
print(static_audit(cases["contínua + tab10"], profile).feedback())

Os testes (`tests/test_audit.py`) cobrem casos que **têm** de ser rejeitados, casos que **têm** de passar, avisos,
e a camada opcional **LLM-as-judge**. Para os correr: `python -m pytest`.

In [ ]:
import subprocess
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", "--color=no"], capture_output=True, text=True, cwd=ROOT)
print(res.stdout[-600:])

## 5. Grafo cíclico LangGraph
```
START → generate → audit ──(aprovado)──→ execute ──(ok)──→ END
           ↑          │                      │
           └──────────┴─(reprovado / erro)───┘      (limite de iterações → give_up → END)
```
- **generate** — o LLM escreve (ou reescreve) o código, recebendo o *feedback* do auditor ou o *traceback*.
- **audit** — valida as escalas de cor **antes** de executar (barato e determinístico).
- **execute** — corre o código num subprocesso com *timeout*; se falhar, o erro volta ao gerador.
- **Controlo de erros:** `max_iterations`, *timeout*, *retry* com *backoff* nas falhas da API, e saída controlada (`give_up`) sem exceções.
- **Registo:** cada iteração fica guardada em `outputs/logs/*.json` (evidência para o relatório).

In [ ]:
from src.agents.graph import build_graph
class _NoLLM:
    def invoke(self, *a, **k): raise RuntimeError
print(build_graph(_NoLLM()).get_graph().draw_mermaid())

### Demonstração offline (LLM simulado)
Um LLM simulado devolve primeiro um código **com a paleta errada** (`tab10` numa variável contínua) e depois o corrigido.
Assim vê-se o ciclo sem gastar quota da API.

In [ ]:
from types import SimpleNamespace
from src.agents.graph import run_request

GOOD = '''import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("data/gapminder_clean.csv"); d = df[df.year == 2007]
fig, ax = plt.subplots(figsize=(7, 4.5))
sc = ax.scatter(d.gdpPercap, d.lifeExp, c=d.lifeExp, cmap="viridis")
ax.set_xscale("log"); ax.set(xlabel="PIB per capita (USD, log)", ylabel="Esperança de vida (anos)", title="2007")
fig.colorbar(sc, label="Esperança de vida (anos)")
plt.savefig(OUTPUT_PATH, bbox_inches="tight")
'''
BAD = GOOD.replace('cmap="viridis"', 'cmap="tab10"')

class FakeLLM:
    def __init__(self, *codes): self.codes = list(codes)
    def invoke(self, messages):
        code = self.codes.pop(0) if len(self.codes) > 1 else self.codes[0]
        return SimpleNamespace(content=f"```python\n{code}```")

out = run_request("Scatter colorido por lifeExp", FakeLLM(BAD, GOOD), label="notebook_offline")
print("estado:", out["status"], "| iterações:", out["iterations"])
for h in out["history"]:
    print(f"  it.{h['iteration']} {h['stage']:7s}", {k: v for k, v in h.items() if k in ('approved', 'ok')},
          [f['rule'] for f in h.get('findings', [])])
display(Image(out["figure_path"]))

### Execução com o Gemini real
Modelo configurável por `GEMINI_MODEL` (por omissão `gemini-3.1-flash-lite`).

In [ ]:
out = run_request("Scatter de PIB per capita vs esperança de vida em 2007, cor por continente, tamanho por população.",
                  label="notebook_real")
print("estado:", out["status"], "| iterações:", out["iterations"])
if out.get("figure_path"):
    display(Image(out["figure_path"]))
else:
    print("erro:", out.get("error"))

## 6. Cenários de autocorreção (LLM real)
`python -m src.scenarios` pede ao gerador gráficos que **violam** as regras de cor (R1, R2, R4, R5) e um pedido válido.
Abaixo, os resultados guardados em `outputs/scenarios_results.json`.

In [ ]:
res_path = ROOT / "outputs" / "scenarios_results.json"
if res_path.exists():
    res = pd.DataFrame(json.loads(res_path.read_text(encoding="utf-8")))
    display(res[["scenario", "expected_rule", "status", "iterations", "first_audit_approved", "first_audit_rules"]])
else:
    print("Ainda sem resultados: correr  python -m src.scenarios")

## 7. Conclusões
- Os mapeamentos (posição, matiz, forma, tamanho) e as escalas (linear vs. log) foram escolhidos e **justificados** com base nos dados (assimetria) e em Wilke.
- O **auditor de cor** formaliza a regra «tipo de paleta ↔ natureza da variável» e é testável (`pytest`).
- O **grafo LangGraph** fecha o ciclo: o auditor e o executor devolvem feedback ao gerador, que reescreve o código de forma autónoma, com limites e saída controlada em caso de falha.
- **Limitação:** o auditor valida cor; outras boas práticas (legendas legíveis, formatação de números) dependem do prompt do gerador.

*Parte II (séries temporais, daltonismo, integridade dos eixos) e Parte I Fase 2 estão fora do âmbito deste notebook.*